In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import DataLoader, TensorDataset
import joblib
import os

In [ ]:
# Get the directory of the current notebook (if it's in the same folder as the CSVs)
project_dir = os.getcwd()
# Now construct the path reliably
normal_csv_path = os.path.join(project_dir, '../data/normal2.csv')
slowed_csv_path = os.path.join(project_dir, '../data/slowed2.csv')

In [ ]:
WINDOW_SIZE = 128

def load_current(file_path):
    df = pd.read_csv(file_path, header=None, names=['raw', 'baseline', 'delta', 'current'])
    return df['current'].values.reshape(-1, 1)

def make_windows(data, label_val, window_size=WINDOW_SIZE):
    windows, labels = [], []
    for i in range(len(data) - window_size):
        windows.append(data[i:i + window_size].T)  # Shape [1, window_size]
        labels.append(label_val)
    return torch.tensor(np.array(windows), dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

In [ ]:
# Load raw current readings
normal_raw = load_current(normal_csv_path)
slowed_raw = load_current(slowed_csv_path)

# Fit ONE scaler on data from both classes. Fitting a separate MinMaxScaler
# per file (as before) lets the model "cheat" by learning per-file scale
# artifacts instead of the real normal/slowed difference in the signal.
scaler = MinMaxScaler()
scaler.fit(np.vstack([normal_raw, slowed_raw]))

normal_scaled = scaler.transform(normal_raw)
slowed_scaled = scaler.transform(slowed_raw)

X_n, y_n = make_windows(normal_scaled, 0)
X_s, y_s = make_windows(slowed_scaled, 1)
X = torch.cat([X_n, X_s], dim=0)
y = torch.cat([y_n, y_s], dim=0)

# Hold out a test split so we can tell whether the model actually learned
# anything, instead of only ever seeing training-set loss.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

In [ ]:
class MotorFaultCNN(nn.Module):
    def __init__(self):
        super(MotorFaultCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5)
        self.bn1 = nn.BatchNorm1d(32) # Added
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(64) # Added
        self.fc = nn.Linear(64 * 30, 2)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [ ]:
model = MotorFaultCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
print("Starting training...")
for epoch in range(10): # 10 epochs
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# Evaluate on the held-out test split -- this is the number that actually
# tells you whether the model generalizes, not just the training loss above.
model.eval()
with torch.no_grad():
    test_output = model(X_test)
    test_preds = torch.argmax(test_output, dim=1)

acc = accuracy_score(y_test, test_preds)
print(f"Test accuracy: {acc:.2%}")
print(classification_report(y_test, test_preds, target_names=["normal", "slowed"]))

In [ ]:
# Save the model AND the scaler it was trained with.
# The scaler MUST be reused at inference time (see the live-inference cell
# below) or the model will see differently-scaled input than it was
# trained on, which silently produces wrong predictions.
torch.save(model.state_dict(), "../models/motor_model_scratch.pth") # uncomment this line to update the main model
joblib.dump(scaler, "../models/motor_scaler.joblib")
print("Model saved as motor_model_scratch.pth")
print("Scaler saved as motor_scaler.joblib")

In [ ]:
import serial
import numpy as np
import time

# Update with your Pico 2 port (e.g., /dev/ttyACM0)
ser = serial.Serial('/dev/ttyACM0', 115200, timeout=1)

def get_live_window():
    window = []
    print("Collecting 128 samples...")
    while len(window) < 128:
        line = ser.readline().decode('utf-8').strip()
        try:
            # Assuming line is "raw,baseline,delta,current"
            parts = line.split(',')
            if len(parts) >= 4:
                val = float(parts[3]) # This is your 'current'
                window.append(val)
        except:
            continue
    return np.array(window)

live_window = get_live_window()

In [ ]:
# inference.py
import torch
import numpy as np
import joblib

# 1. Load the model and the EXACT scaler used at training time
model = MotorFaultCNN() # Ensure this class is defined exactly as in your training script
model.load_state_dict(torch.load("../models/motor_model_scratch.pth"))
model.eval()

scaler = joblib.load("../models/motor_scaler.joblib")

# 2. Simulate or capture live data
def classify_live_data(live_window):
    # Use the SAME scaler fitted during training instead of a guessed
    # constant. A mismatched scaler here is the most common way this kind
    # of model silently gives wrong answers on live hardware.
    data_scaled = scaler.transform(np.array(live_window).reshape(-1, 1))
    data = data_scaled.reshape(1, 1, len(live_window))
    data = torch.tensor(data, dtype=torch.float32)

    with torch.no_grad():
        output = model(data)
        prediction = torch.argmax(output, dim=1).item()

    return "Normal" if prediction == 0 else "Slowed"
# Test it!
print(f"Prediction: {classify_live_data(live_window)}")